# Week 5 — Deep Learning Application in Data Science
## Fashion-MNIST Image Classification using a Convolutional Neural Network
**Platform:** YuvaIntern / NSDC  
**Framework:** PyTorch  
**Dataset:** Fashion-MNIST training CSV (60,000 images, 10 classes)

## 1. Project Objectives
- Apply a deep learning model to a real image-classification problem.
- Preprocess and visualize image data.
- Design, train and evaluate a CNN.
- Compare the CNN with a fully connected neural-network baseline.
- Analyze class-wise errors, overfitting and practical resource constraints.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

CSV_PATH = "fashion-mnist_train.csv"
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print(df.head())
print(df.isna().sum().sum(), "missing values")
print("Duplicate rows:", df.duplicated().sum())


## 2. Exploratory Data Analysis
Each row contains one 28×28 grayscale image represented by 784 pixel columns. The label column contains one of ten clothing categories.

In [ ]:
X = df.drop(columns=["label"]).values.astype("float32") / 255.0
y = df["label"].values.astype("int64")
X = X.reshape(-1, 1, 28, 28)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=512, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
    batch_size=1024, shuffle=False
)
print("Training:", X_train.shape, "Validation:", X_val.shape)


## 3. Preprocessing
Pixel intensities are scaled from 0–255 to 0–1. Images are reshaped to `(1, 28, 28)` and the data is split into 80% training and 20% validation using stratification.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

model = CNN()
print(model)


## 4. CNN Architecture
Three convolutional blocks progressively learn local-to-global image features. Max-pooling reduces spatial size. A 128-unit dense layer performs high-level classification, while dropout reduces overfitting.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
best_state = None

for epoch in range(5):
    model.train()
    total_loss = correct = total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        correct += (out.argmax(1) == yb).sum().item()
        total += len(yb)

    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)

    val_loss = total_loss / total
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch+1}/5 | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

model.load_state_dict(best_state)


## 5. Training Configuration
Optimizer: Adam  
Learning rate: 0.003  
Loss: Cross-Entropy  
Batch size: 512  
Epochs: 5  
Random seed/split: 42  
The model was intentionally kept compact because the available execution environment was CPU-only.

In [ ]:
model.eval()
pred, true = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        out = model(xb.to(device))
        pred.extend(out.argmax(1).cpu().numpy())
        true.extend(yb.numpy())

pred = np.array(pred)
true = np.array(true)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Accuracy:", accuracy_score(true, pred))
print(classification_report(true, pred, target_names=class_names))
print(confusion_matrix(true, pred))


## 6. Recorded Results
The completed run reached **87.98% validation accuracy**, with weighted precision **88.51%**, weighted recall **87.98%**, and weighted F1-score **88.14%**. The baseline fully connected model reached **86.58%**, so the CNN improved validation accuracy by approximately **1.39 percentage points**.

## 7. Interpretation
The confusion matrix shows that the hardest category is Shirt, with frequent confusion against T-shirt/top, Pullover and Coat. Trouser, Sandal, Bag and Sneaker are comparatively easier because their silhouettes are more distinctive. The relatively close training and validation accuracy at the end of training indicates that severe overfitting was not observed in the five-epoch run.

## 8. Reproducibility
Place `fashion-mnist_train.csv` in the same directory as this notebook before running it. Install dependencies with `pip install pandas numpy matplotlib scikit-learn torch` if necessary.